In [5]:
import sys
print(sys.executable)

c:\Users\Dell\Documents\EV_Charging_Demand_Analysis\.venv\Scripts\python.exe


In [ ]:

# ==========================================
# EV CHARGING DEMAND CLASSIFICATION
# MODEL COMPARISON
# ==========================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

# STEP 1: LOAD DATA
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT / "data" / "processed"
    / "ChargingRecords_Cleaned.csv"
)

df = pd.read_csv(DATA_PATH)

df["StartDatetime"] = pd.to_datetime(
    df["StartDatetime"], errors="coerce"
)

df = df.dropna(
    subset=["StartDatetime", "Demand"]
).copy()

# STEP 2: CREATE TARGET
threshold = df["Demand"].median()

df["DemandClass"] = np.where(
    df["Demand"] >= threshold,
    "High",
    "Normal"
)

print("Median demand threshold:", threshold)
print("\nClass distribution:")
print(df["DemandClass"].value_counts())

print("\nClass percentages:")
print(
    (df["DemandClass"].value_counts(normalize=True) * 100)
    .round(2)
)

# STEP 3: CREATE FEATURES
df["StartHour"] = df["StartDatetime"].dt.hour
df["StartMonth"] = df["StartDatetime"].dt.month
df["StartDayOfWeek"] = df["StartDatetime"].dt.dayofweek
df["StartDayOfMonth"] = df["StartDatetime"].dt.day
df["IsWeekend"] = (df["StartDayOfWeek"] >= 5).astype(int)

feature_columns = [
    "ChargerCompany",
    "Location",
    "ChargerType",
    "StartHour",
    "StartMonth",
    "StartDayOfWeek",
    "StartDayOfMonth",
    "IsWeekend"
]

X = df[feature_columns].copy()
y = df["DemandClass"].copy()

X = pd.get_dummies(
    X,
    columns=["Location"],
    drop_first=False,
    dtype=int
)

# STEP 4: TRAIN/TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining shape:", X_train.shape)
print("Testing shape:", X_test.shape)

# STEP 5: DEFINE MODELS
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42)
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=15,
        min_samples_leaf=2,
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1
    ),

    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
    ),

    "SVM": make_pipeline(
    StandardScaler(),
    SVC(kernel="linear", probability=False, random_state=42)
    )
    
}

# STEP 6: TRAIN AND EVALUATE
results = []
trained_models = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)
    trained_models[name] = model

    predictions = model.predict(X_test)

    high_index = list(model.classes_).index("High") \
        if hasattr(model, "classes_") else \
        list(model[-1].classes_).index("High")

    if hasattr(model, "predict_proba"):
        if hasattr(model[-1] if hasattr(model, "steps") else model, "decision_function"):
           scores = model.decision_function(X_test)
           classes = model.classes_ if hasattr(model, "classes_") else model[-1].classes_
           high_index = list(classes).index("High")
           if high_index == 1:
            probabilities = scores
           else:
            probabilities = -scores
        else:
          probabilities = model.predict_proba(X_test)[:, high_index]
    else:
        probabilities = None

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions,
            pos_label="High", zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions,
            pos_label="High", zero_division=0
        ),
        "F1": f1_score(
            y_test, predictions,
            pos_label="High", zero_division=0
        ),
        "ROC_AUC": (
            roc_auc_score(
                (y_test == "High").astype(int),
                probabilities
            )
            if probabilities is not None else np.nan
        )
    }

    results.append(row)

classification_results = pd.DataFrame(results)
classification_results = classification_results.sort_values(
    "F1", ascending=False
)

print("\nCLASSIFICATION MODEL COMPARISON")
print(classification_results.round(4).to_string(index=False))

# STEP 7: BEST MODEL REPORT
best_model_name = classification_results.iloc[0]["Model"]
best_model = trained_models[best_model_name]

print("\nBest model by F1:", best_model_name)
print("\nClassification report:")
print(
    classification_report(
        y_test,
        best_model.predict(X_test),
        zero_division=0
    )
)

# STEP 8: SAVE RESULTS
REPORT_PATH = PROJECT_ROOT / "reports"
REPORT_PATH.mkdir(parents=True, exist_ok=True)

classification_results.to_csv(
    REPORT_PATH / "classification_model_comparison.csv",
    index=False
)

print("\nResults saved to reports/classification_model_comparison.csv")

Median demand threshold: 14.1

Class distribution:
DemandClass
High      36439
Normal    36387
Name: count, dtype: int64

Class percentages:
DemandClass
High      50.04
Normal    49.96
Name: proportion, dtype: float64

Training shape: (58260, 21)
Testing shape: (14566, 21)

Training Logistic Regression...

Training Decision Tree...

Training Random Forest...

Training KNN...

Training SVM...


c:\Users\Dell\Documents\EV_Charging_Demand_Analysis\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
